<a href="https://colab.research.google.com/github/KrizChin/AI-Plant-Diagnostic-Tool/blob/main/Plant_AI.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [3]:
# import dataset
import kagglehub

# Download latest version
path = kagglehub.dataset_download("emmarex/plantdisease")

print("Path to dataset files:", path)

Using Colab cache for faster access to the 'plantdisease' dataset.
Path to dataset files: /kaggle/input/plantdisease


In [4]:
# import kaggle api key
from google.colab import userdata
userdata.get('KAGGLE')

'KGAT_810971d91d7950f3fd1fec5954e8efef'

In [5]:
# add file paths
from pathlib import Path
import shutil
import os

source_dir = Path("/kaggle/input/plantdisease/PlantVillage")
tomato_dir = Path("/content/tomato_dataset")

tomato_dir.mkdir(exist_ok=True)

print("Source exists:", source_dir.exists())

Source exists: True


In [6]:
# copy tomato folders
for class_folder in source_dir.iterdir():
    if class_folder.is_dir() and class_folder.name.startswith("Tomato"):
        destination = tomato_dir / class_folder.name

        if not destination.exists():
            shutil.copytree(class_folder, destination)

print("Tomato folders copied:")
print(os.listdir(tomato_dir))

Tomato folders copied:
['Tomato__Tomato_mosaic_virus', 'Tomato_healthy', 'Tomato_Spider_mites_Two_spotted_spider_mite', 'Tomato__Tomato_YellowLeaf__Curl_Virus', 'Tomato_Bacterial_spot', 'Tomato_Late_blight', 'Tomato_Early_blight', 'Tomato_Septoria_leaf_spot', 'Tomato_Leaf_Mold', 'Tomato__Target_Spot']


In [7]:
import tensorflow as tf

img_size = (224, 224)
batch_size = 32

train_ds = tf.keras.utils.image_dataset_from_directory(
    tomato_dir,
    validation_split=0.2,
    subset="training",
    seed=123,
    image_size=img_size,
    batch_size=batch_size
)

val_ds = tf.keras.utils.image_dataset_from_directory(
    tomato_dir,
    validation_split=0.2,
    subset="validation",
    seed=123,
    image_size=img_size,
    batch_size=batch_size
)

class_names = train_ds.class_names
print("Classes:")
print(class_names)

Found 16011 files belonging to 10 classes.
Using 12809 files for training.
Found 16011 files belonging to 10 classes.
Using 3202 files for validation.
Classes:
['Tomato_Bacterial_spot', 'Tomato_Early_blight', 'Tomato_Late_blight', 'Tomato_Leaf_Mold', 'Tomato_Septoria_leaf_spot', 'Tomato_Spider_mites_Two_spotted_spider_mite', 'Tomato__Target_Spot', 'Tomato__Tomato_YellowLeaf__Curl_Virus', 'Tomato__Tomato_mosaic_virus', 'Tomato_healthy']


In [ ]:
from tensorflow.keras import layers, models

num_classes = len(class_names)

model = models.Sequential([
    layers.Rescaling(1./255, input_shape=(224, 224, 3)),

    layers.Conv2D(16, 3, activation="relu"),
    layers.MaxPooling2D(),

    layers.Conv2D(32, 3, activation="relu"),
    layers.MaxPooling2D(),

    layers.Conv2D(64, 3, activation="relu"),
    layers.MaxPooling2D(),

    layers.Flatten(),
    layers.Dense(128, activation="relu"),
    layers.Dense(num_classes, activation="softmax")
])

model.compile(
    optimizer="adam",
    loss="sparse_categorical_crossentropy",
    metrics=["accuracy"]
)

history = model.fit(
    train_ds,
    validation_data=val_ds,
    epochs=5
)

/usr/local/lib/python3.12/dist-packages/keras/src/layers/preprocessing/data_layer.py:95: UserWarning: Do not pass an `input_shape`/`input_dim` argument to a layer. When using Sequential models, prefer using an `Input(shape)` object as the first layer in the model instead.
  super().__init__(**kwargs)


Epoch 1/5
401/401 ━━━━━━━━━━━━━━━━━━━━ 682s 2s/step - accuracy: 0.6807 - loss: 0.9297 - val_accuracy: 0.7942 - val_loss: 0.6081
Epoch 2/5
401/401 ━━━━━━━━━━━━━━━━━━━━ 673s 2s/step - accuracy: 0.8638 - loss: 0.3969 - val_accuracy: 0.8379 - val_loss: 0.4683
Epoch 3/5
401/401 ━━━━━━━━━━━━━━━━━━━━ 628s 2s/step - accuracy: 0.9147 - loss: 0.2444 - val_accuracy: 0.8598 - val_loss: 0.3988
Epoch 4/5
401/401 ━━━━━━━━━━━━━━━━━━━━ 711s 2s/step - accuracy: 0.9468 - loss: 0.1532 - val_accuracy: 0.8538 - val_loss: 0.5199
Epoch 5/5
388/401 ━━━━━━━━━━━━━━━━━━━━ 18s 1s/step - accuracy: 0.9520 - loss: 0.1303

In [9]:
loss, accuracy = model.evaluate(val_ds)

print("Validation accuracy:", round(accuracy * 100, 2), "%")

101/101 ━━━━━━━━━━━━━━━━━━━━ 46s 451ms/step - accuracy: 0.8819 - loss: 0.3648
Validation accuracy: 88.19 %


In [15]:
import random
import numpy as np
from tensorflow.keras.utils import load_img, img_to_array

# Pick a random class folder
random_class = random.choice(os.listdir(tomato_dir))
class_path = tomato_dir / random_class

# Pick random image from that folder
random_image = random.choice(os.listdir(class_path))
test_image_path = class_path / random_image

print("Testing image:", test_image_path)
print("Actual class:", random_class)

img = load_img(test_image_path, target_size=(224, 224))
img_array = img_to_array(img)
img_array = np.expand_dims(img_array, axis=0)

predictions = model.predict(img_array)

predicted_class = class_names[np.argmax(predictions[0])]
confidence = 100 * np.max(predictions[0])

print("Predicted class:", predicted_class)
print("Confidence:", round(confidence, 2), "%")

Testing image: /content/tomato_dataset/Tomato__Tomato_mosaic_virus/570486a0-abe0-4046-add9-14aab37de620___PSU_CG 2313.JPG
Actual class: Tomato__Tomato_mosaic_virus
1/1 ━━━━━━━━━━━━━━━━━━━━ 0s 50ms/step
Predicted class: Tomato_Septoria_leaf_spot
Confidence: 56.7 %


In [11]:
advice = {
    "Tomato_Bacterial_spot": "Remove infected leaves, avoid overhead watering, and improve air circulation.",
    "Tomato_Early_blight": "Remove lower infected leaves, avoid splashing soil onto leaves, and use mulch.",
    "Tomato_Late_blight": "Remove infected leaves quickly and separate infected plants if possible.",
    "Tomato_Leaf_Mold": "Improve air circulation and reduce humidity around the plant.",
    "Tomato_Septoria_leaf_spot": "Remove infected leaves and avoid watering the leaves directly.",
    "Tomato_Spider_mites_Two_spotted_spider_mite": "Spray leaves with water, remove heavily damaged leaves, and check undersides of leaves.",
    "Tomato_Target_Spot": "Remove infected leaves and avoid wet foliage.",
    "Tomato_Tomato_YellowLeaf_Curl_Virus": "Remove infected plants and control whiteflies.",
    "Tomato_Tomato_mosaic_virus": "Remove infected plants and disinfect tools.",
    "Tomato_healthy": "The plant appears healthy. Continue normal care."
}

print("Suggested solution:", advice.get(predicted_class, "No advice available."))

Suggested solution: Remove infected leaves quickly and separate infected plants if possible.
